
# 08 — EDA Quality Report & Feature Selection

**Mục tiêu notebook:** tạo một báo cáo EDA riêng cho đồ án DS108, tập trung vào chất lượng dữ liệu, câu chuyện dữ liệu, missingness, imputation, đặc trưng khí tượng và kiểm chứng feature.

Notebook này được xây dựng bằng cách **giữ lại và mở rộng nội dung từ notebook bạn đã đính kèm** `eda_feature_selection_ngan_gon(1).ipynb`, gồm các phần:
- Phân tích biến mục tiêu.
- Phân tích mùa vụ.
- Khả năng phân tách lớp bằng histogram/boxplot.
- Mann–Whitney U.
- Mutual Information.
- Spearman correlation và đa cộng tuyến.
- Random Forest importance.
- Chọn feature cuối cùng và đánh giá nhanh.

Notebook mới bổ sung các phần còn thiếu theo guideline:
- Missingness heatmap theo biến và theo thời gian.
- Tỷ lệ missing theo trạm.
- Phân phối `PRCP`, `TEMP`, `DEWP`, `SLP`, `ENSO`.
- So sánh GSOD và ERA5 ở các ngày trùng.
- Mưa theo tháng, trạm/khu vực.
- Correlation heatmap.
- Class imbalance.
- Trước/sau imputation: mean, std, distribution shift.
- Top feature importance sau model.

> **Lưu ý chống leakage:**  
> `PRCP`, `PRCP_mm`, `target_prcp_mm` là các biến liên quan trực tiếp đến nhãn mưa nên **không được dùng làm input feature** khi đánh giá mô hình. Target chính của pipeline mới là `rain_target`. Nếu notebook chỉ tìm thấy `Target`, nó sẽ dùng fallback để tương thích với notebook cũ.


## 0. Thiết lập môi trường, đường dẫn và helper functions

In [ ]:

import os
import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

from scipy.stats import mannwhitneyu

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef
)

warnings.filterwarnings("ignore")
plt.rcParams["axes.unicode_minus"] = False

if HAS_SEABORN:
    sns.set_theme(style="whitegrid", font_scale=1.0)

# ----------------------------------------------------------------------------
# Project paths
# ----------------------------------------------------------------------------
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "data").exists() or (p / "reports").exists():
            return p
    return Path.cwd()

BASE_DIR = find_project_root()
OUTPUT_DIR = BASE_DIR / "reports" / "eda_quality_report"
PLOT_DIR = OUTPUT_DIR / "plots"
TABLE_DIR = OUTPUT_DIR / "tables"

for d in [OUTPUT_DIR, PLOT_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Ưu tiên file full feature-engineered để còn audit các source/imputation columns.
DATA_CANDIDATES = [
    BASE_DIR / "data" / "feature_engineering" / "feature_engineered_data.csv",
    BASE_DIR / "data" / "feature_engineering" / "model_ready_data.csv",
    BASE_DIR / "data" / "clean" / "silver_data_ver2.csv",
    Path("feature_engineered_data.csv"),
    Path("model_ready_data.csv"),
    Path("silver_data_ver2.csv"),
]

SILVER_REPORT_DIR = BASE_DIR / "reports" / "data_quality" / "silver"
MODEL_REPORT_DIR = BASE_DIR / "reports" / "model_validation"
FEATURE_REPORT_DIR = BASE_DIR / "reports" / "data_quality" / "feature_engineering"

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:

# ----------------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------------
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def save_table(df, name):
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"[Saved table] {path}")
    return path

def save_fig(name):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[Saved figure] {path}")
    plt.show()
    return path

def display_section_note(text):
    display(Markdown(f"> {text}"))

def numeric_summary(df, cols):
    rows = []
    for col in cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        rows.append({
            "column": col,
            "count": int(s.notna().sum()),
            "missing_count": int(s.isna().sum()),
            "missing_rate_%": round(s.isna().mean() * 100, 3),
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(),
            "min": s.min(),
            "p05": s.quantile(0.05),
            "p25": s.quantile(0.25),
            "p75": s.quantile(0.75),
            "p95": s.quantile(0.95),
            "max": s.max(),
        })
    return pd.DataFrame(rows)

def safe_target_col(df):
    if "rain_target" in df.columns:
        return "rain_target"
    if "Target" in df.columns:
        return "Target"
    return None

def safe_prcp_col(df):
    if "PRCP_mm" in df.columns:
        return "PRCP_mm"
    if "PRCP" in df.columns:
        return "PRCP"
    if "target_prcp_mm" in df.columns:
        return "target_prcp_mm"
    return None

def plot_hist(series, title, xlabel, filename, log1p=False):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        print(f"Skip {title}: no valid data")
        return
    if log1p:
        s = np.log1p(s)
        xlabel = "log(1 + " + xlabel + ")"
    plt.figure(figsize=(8, 4.8))
    plt.hist(s, bins=50)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    save_fig(filename)


## 1. Load dữ liệu và xác định biến mục tiêu

In [ ]:

DATA_PATH = first_existing(DATA_CANDIDATES)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy dữ liệu. Hãy chạy Step 5/6 trước hoặc đặt file "
        "feature_engineered_data.csv/model_ready_data.csv/silver_data_ver2.csv trong project."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], errors="coerce")

TARGET_COL = safe_target_col(df)
PRCP_COL = safe_prcp_col(df)

# Fallback: nếu chưa có target nhưng có PRCP, tạo target chỉ cho EDA.
if TARGET_COL is None and PRCP_COL is not None:
    TARGET_COL = "rain_target"
    df[TARGET_COL] = (pd.to_numeric(df[PRCP_COL], errors="coerce") > 0.1).astype(int)
    print("Created fallback rain_target from", PRCP_COL)

if TARGET_COL is None:
    raise ValueError("Không tìm thấy target. Cần có rain_target, Target hoặc PRCP/PRCP_mm để tạo target.")

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").astype("Int64")

print("TARGET_COL:", TARGET_COL)
print("PRCP_COL:", PRCP_COL)

if "time" in df.columns:
    print("Date range:", df["time"].min(), "→", df["time"].max())
if "STATION" in df.columns:
    print("Number of stations:", df["STATION"].nunique())

display(df.head())



## 2. Kiểm tra chất lượng dữ liệu tổng quan

Mục tiêu phần này là trả lời:
- Dataset có bao nhiêu dòng/cột?
- Kiểu dữ liệu có hợp lý không?
- Có duplicate theo dòng hoặc theo `STATION + time` không?
- Biến nào thiếu nhiều nhất?
- Có dấu hiệu leakage rõ ràng không?


In [ ]:

quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_rate_%": (df.isna().mean() * 100).round(3),
    "n_unique": df.nunique(dropna=True)
}).sort_values("missing_rate_%", ascending=False)

print("Số dòng, số cột:", df.shape)
print("Duplicate toàn dòng:", df.duplicated().sum())

if {"STATION", "time"}.issubset(df.columns):
    print("Duplicate theo STATION + time:", df.duplicated(subset=["STATION", "time"]).sum())

display(quality_summary.head(30))
save_table(quality_summary.reset_index().rename(columns={"index": "column"}), "01_data_quality_summary.csv")

# Leakage warning table
leakage_candidates = [
    c for c in ["PRCP", "PRCP_mm", "target_prcp_mm", "target_time", "rain_target", "Target"]
    if c in df.columns
]
leakage_table = pd.DataFrame({
    "column": leakage_candidates,
    "reason": [
        "Target hoặc biến liên quan trực tiếp đến target; không dùng làm input feature."
        for _ in leakage_candidates
    ]
})
display(leakage_table)
save_table(leakage_table, "02_leakage_candidates.csv")



## 3. Missingness analysis

Guideline yêu cầu không xử lý missing một cách cơ học. Vì vậy notebook phân tích:
- Missing rate theo biến.
- Missing rate theo trạm.
- Missingness theo thời gian.
- Heatmap missingness theo tháng và biến.

Nếu bạn đã chạy Step 5 improved, notebook cũng đọc lại các report missingness trước imputation trong `reports/data_quality/silver/`.


In [ ]:

# 3.1 Missing rate by variable
missing_by_col = (
    df.isna().mean()
    .mul(100)
    .sort_values(ascending=True)
    .rename("missing_rate_%")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_by_col.tail(30))
save_table(missing_by_col, "03_missing_rate_by_column_current_data.csv")

plt.figure(figsize=(9, max(5, min(14, len(missing_by_col) * 0.22))))
plt.barh(missing_by_col["column"].tail(40), missing_by_col["missing_rate_%"].tail(40))
plt.xlabel("Missing rate (%)")
plt.title("Top missing variables — current dataset")
save_fig("03_missing_rate_by_column_current_data.png")


In [ ]:

# 3.2 Missing rate by station
if "STATION" in df.columns:
    key_vars_for_missing = [
        c for c in ["PRCP", "PRCP_mm", "TEMP", "DEWP", "SLP", "STP", "VISIB", "ENSO", "ENSO_lag_1"]
        if c in df.columns
    ]
    station_missing = df.groupby("STATION")[key_vars_for_missing].apply(lambda x: x.isna().mean() * 100)
    station_missing = station_missing.reset_index()
    display(station_missing)
    save_table(station_missing, "04_missing_rate_by_station.csv")

    plot_df = station_missing.set_index("STATION")
    plt.figure(figsize=(10, max(4, 0.6 * len(plot_df))))
    if HAS_SEABORN:
        sns.heatmap(plot_df, annot=True, fmt=".1f", cmap="viridis")
    else:
        plt.imshow(plot_df.values, aspect="auto")
        plt.yticks(range(len(plot_df.index)), plot_df.index)
        plt.xticks(range(len(plot_df.columns)), plot_df.columns, rotation=45, ha="right")
        plt.colorbar(label="Missing rate (%)")
    plt.title("Missing rate theo trạm (%)")
    save_fig("04_missing_rate_by_station_heatmap.png")
else:
    print("Không có cột STATION, bỏ qua missingness theo trạm.")


In [ ]:

# 3.3 Missingness over time: month x variable heatmap
if "time" in df.columns:
    time_df = df.copy()
    time_df["year_month"] = time_df["time"].dt.to_period("M").astype(str)

    key_vars = [
        c for c in ["PRCP", "PRCP_mm", "TEMP", "DEWP", "SLP", "STP", "VISIB", "ENSO", "ENSO_lag_1"]
        if c in time_df.columns
    ]

    if key_vars:
        monthly_missing = time_df.groupby("year_month")[key_vars].apply(lambda x: x.isna().mean() * 100)
        display(monthly_missing.head())
        save_table(monthly_missing.reset_index(), "05_monthly_missingness.csv")

        plt.figure(figsize=(12, max(4, 0.25 * len(monthly_missing))))
        if HAS_SEABORN:
            sns.heatmap(monthly_missing, cmap="viridis")
        else:
            plt.imshow(monthly_missing.values, aspect="auto")
            plt.yticks(range(len(monthly_missing.index)), monthly_missing.index)
            plt.xticks(range(len(monthly_missing.columns)), monthly_missing.columns, rotation=45, ha="right")
            plt.colorbar(label="Missing rate (%)")
        plt.title("Missingness heatmap theo tháng và biến")
        plt.xlabel("Variable")
        plt.ylabel("Year-Month")
        save_fig("05_monthly_missingness_heatmap.png")
else:
    print("Không có cột time, bỏ qua missingness theo thời gian.")


In [ ]:

# 3.4 Đọc report missingness trước imputation từ Step 5 nếu có
silver_missing_reports = [
    SILVER_REPORT_DIR / "missing_rate_before_imputation_by_column.csv",
    SILVER_REPORT_DIR / "missing_rate_before_imputation_by_station.csv",
    SILVER_REPORT_DIR / "missing_rate_before_imputation_by_month.csv",
    SILVER_REPORT_DIR / "missingness_indicator_signal_correlation.csv",
]

for report_path in silver_missing_reports:
    if report_path.exists():
        print(f"\nLoaded Step 5 report: {report_path.name}")
        rep = pd.read_csv(report_path)
        display(rep.head(15))
    else:
        print(f"Not found: {report_path}")



**Nhận xét cần viết trong báo cáo:**  
Nếu missingness tập trung ở một số trạm hoặc giai đoạn nhất định, dữ liệu có khả năng không phải MCAR hoàn toàn. Khi đó, việc dùng ERA5 để bù khuyết có cơ sở hơn so với xóa dòng, vì xóa dòng có thể làm mất cấu trúc mùa vụ hoặc làm lệch phân phối theo vùng.


## 4. Phân tích phân bố các biến khí tượng chính

In [ ]:

core_vars = [c for c in ["PRCP", "PRCP_mm", "TEMP", "DEWP", "SLP", "STP", "ENSO", "ENSO_lag_1", "ENSO_lag_2"] if c in df.columns]
core_summary = numeric_summary(df, core_vars)
display(core_summary)
save_table(core_summary, "06_core_variable_summary.csv")

for col in core_vars:
    log1p = col in ["PRCP", "PRCP_mm"]
    plot_hist(
        df[col],
        title=f"Distribution of {col}",
        xlabel=col,
        filename=f"06_distribution_{col}.png",
        log1p=log1p
    )


In [ ]:

# Boxplot theo target cho các biến chính
if TARGET_COL in df.columns:
    plot_vars = [c for c in ["TEMP", "DEWP", "SLP", "STP", "ENSO", "ENSO_lag_1"] if c in df.columns]
    for col in plot_vars:
        plot_df = df[[TARGET_COL, col]].dropna()
        if plot_df.empty:
            continue
        plt.figure(figsize=(7, 4.5))
        if HAS_SEABORN:
            sns.boxplot(data=plot_df, x=TARGET_COL, y=col)
        else:
            groups = [plot_df.loc[plot_df[TARGET_COL] == v, col] for v in sorted(plot_df[TARGET_COL].dropna().unique())]
            plt.boxplot(groups, labels=sorted(plot_df[TARGET_COL].dropna().unique()))
        plt.title(f"{col} theo lớp mưa/không mưa")
        plt.xlabel(f"{TARGET_COL}: 0 = no rain, 1 = rain")
        plt.ylabel(col)
        save_fig(f"07_boxplot_{col}_by_target.png")



## 5. Phân tích biến mục tiêu và mất cân bằng lớp

Phần này kế thừa nội dung từ notebook cũ: kiểm tra tỷ lệ ngày mưa/không mưa, sau đó phân tích theo trạm để xem class imbalance có khác nhau theo không gian hay không.


In [ ]:

class_counts = df[TARGET_COL].value_counts(dropna=False).sort_index()
class_rate = (class_counts / class_counts.sum() * 100).round(2)

target_table = pd.DataFrame({
    "class": class_counts.index.astype(str),
    "count": class_counts.values,
    "rate_%": class_rate.values
})
display(target_table)
save_table(target_table, "08_target_distribution.csv")

plt.figure(figsize=(7, 5))
plt.bar(target_table["class"], target_table["count"])
plt.title("Phân bố lớp mục tiêu")
plt.xlabel(f"{TARGET_COL}: 0 = Không mưa, 1 = Có mưa")
plt.ylabel("Số lượng mẫu")

for i, row in target_table.iterrows():
    plt.text(i, row["count"], f'{row["rate_%"]:.1f}%', ha="center", va="bottom", fontweight="bold")

save_fig("08_target_distribution.png")


In [ ]:

# Rain rate by station/region
if "STATION" in df.columns:
    station_rain = (
        df.groupby("STATION")[TARGET_COL]
        .agg(["count", "mean"])
        .rename(columns={"count": "n_samples", "mean": "rain_rate"})
        .reset_index()
    )
    station_rain["rain_rate_%"] = station_rain["rain_rate"].mul(100).round(2)
    display(station_rain)
    save_table(station_rain, "09_rain_rate_by_station.csv")

    plt.figure(figsize=(10, 5))
    plt.bar(station_rain["STATION"].astype(str), station_rain["rain_rate_%"])
    plt.title("Tỷ lệ ngày mưa theo trạm")
    plt.xlabel("Trạm")
    plt.ylabel("Tỷ lệ ngày mưa (%)")
    plt.xticks(rotation=30)
    save_fig("09_rain_rate_by_station.png")

# Nếu có cột REGION/region thì phân tích thêm
region_col = None
for c in ["REGION", "region", "station_region", "climate_region"]:
    if c in df.columns:
        region_col = c
        break

if region_col:
    region_rain = (
        df.groupby(region_col)[TARGET_COL]
        .agg(["count", "mean"])
        .rename(columns={"count": "n_samples", "mean": "rain_rate"})
        .reset_index()
    )
    region_rain["rain_rate_%"] = region_rain["rain_rate"].mul(100).round(2)
    display(region_rain)
    save_table(region_rain, "10_rain_rate_by_region.csv")
else:
    print("Không có cột region, dùng STATION làm đại diện không gian.")



**Nhận xét cần viết trong báo cáo:**  
Nếu lớp mưa thấp hơn nhiều so với lớp không mưa, Accuracy không đủ tin cậy. Khi đánh giá model nên ưu tiên Recall, Precision, F1-score, PR-AUC và MCC cho class mưa.


## 6. Phân tích mùa vụ mưa theo tháng/năm

In [ ]:

if "time" in df.columns:
    time_df = df.copy()
    time_df["year"] = time_df["time"].dt.year
    time_df["month"] = time_df["time"].dt.month

    monthly_prob = time_df.groupby("month")[TARGET_COL].mean().mul(100).reset_index(name="rain_prob_%")
    display(monthly_prob)
    save_table(monthly_prob, "11_monthly_rain_probability.csv")

    plt.figure(figsize=(11, 5))
    plt.bar(monthly_prob["month"], monthly_prob["rain_prob_%"], alpha=0.75)
    plt.plot(monthly_prob["month"], monthly_prob["rain_prob_%"], marker="o")
    plt.title("Xác suất mưa trung bình theo tháng")
    plt.xlabel("Tháng")
    plt.ylabel("Xác suất mưa (%)")
    plt.xticks(range(1, 13))
    save_fig("11_monthly_rain_probability.png")

    yearly_prob = time_df.groupby("year")[TARGET_COL].mean().mul(100).reset_index(name="rain_prob_%")
    display(yearly_prob)
    save_table(yearly_prob, "12_yearly_rain_probability.csv")

    plt.figure(figsize=(11, 5))
    plt.plot(yearly_prob["year"], yearly_prob["rain_prob_%"], marker="o")
    plt.title("Xác suất mưa trung bình theo năm")
    plt.xlabel("Năm")
    plt.ylabel("Xác suất mưa (%)")
    save_fig("12_yearly_rain_probability.png")
else:
    print("Không có time, bỏ qua seasonality.")


In [ ]:

# Heatmap tháng x trạm
if {"time", "STATION"}.issubset(df.columns):
    time_df = df.copy()
    time_df["month"] = time_df["time"].dt.month
    station_month = time_df.groupby(["STATION", "month"])[TARGET_COL].mean().mul(100).reset_index()
    station_month_pivot = station_month.pivot(index="STATION", columns="month", values=TARGET_COL)
    display(station_month_pivot)
    save_table(station_month, "13_station_month_rain_probability.csv")

    plt.figure(figsize=(12, max(4, 0.6 * station_month_pivot.shape[0])))
    if HAS_SEABORN:
        sns.heatmap(station_month_pivot, annot=True, fmt=".1f", cmap="Blues")
    else:
        plt.imshow(station_month_pivot.values, aspect="auto")
        plt.yticks(range(len(station_month_pivot.index)), station_month_pivot.index)
        plt.xticks(range(len(station_month_pivot.columns)), station_month_pivot.columns)
        plt.colorbar(label="Rain probability (%)")
    plt.title("Xác suất mưa theo trạm và tháng (%)")
    plt.xlabel("Tháng")
    plt.ylabel("Trạm")
    save_fig("13_station_month_rain_probability_heatmap.png")



**Nhận xét cần viết trong báo cáo:**  
Nếu xác suất mưa thay đổi mạnh theo tháng, các đặc trưng chu kỳ như `day_sin`, `day_cos`, `month_sin`, `month_cos` và rolling statistics là hợp lý. Nếu từng trạm có mùa mưa khác nhau, cần phân tích không gian thay vì chỉ dùng thống kê toàn cục.


## 7. So sánh GSOD vs ERA5 và đánh giá trước/sau imputation

In [ ]:

# 7.1 Đọc report GSOD vs ERA5 overlap từ Step 5 nếu có
overlap_path = SILVER_REPORT_DIR / "gsod_vs_era5_overlap_comparison.csv"
if overlap_path.exists():
    overlap = pd.read_csv(overlap_path)
    display(overlap)
    save_table(overlap, "14_gsod_vs_era5_overlap_comparison_copy.csv")

    if {"variable", "spearman_corr"}.issubset(overlap.columns):
        plot_df = overlap.dropna(subset=["spearman_corr"])
        if not plot_df.empty:
            plt.figure(figsize=(8, 4.8))
            plt.bar(plot_df["variable"], plot_df["spearman_corr"])
            plt.title("Spearman correlation giữa GSOD và ERA5 ở ngày trùng")
            plt.xlabel("Variable")
            plt.ylabel("Spearman correlation")
            plt.axhline(0, linewidth=1)
            save_fig("14_gsod_vs_era5_spearman_corr.png")
else:
    display_section_note(
        "Chưa tìm thấy report GSOD vs ERA5. Hãy chạy Step 5 improved để tạo "
        "`reports/data_quality/silver/gsod_vs_era5_overlap_comparison.csv`."
    )


In [ ]:

# 7.2 Trước/sau imputation: mean/std/distribution shift
dist_path = SILVER_REPORT_DIR / "distribution_before_after_imputation.csv"
if dist_path.exists():
    dist_shift = pd.read_csv(dist_path)
    display(dist_shift)
    save_table(dist_shift, "15_distribution_before_after_imputation_copy.csv")

    # Mean shift
    if {"variable", "before_mean", "after_mean"}.issubset(dist_shift.columns):
        plot_df = dist_shift.dropna(subset=["before_mean", "after_mean"]).copy()
        if not plot_df.empty:
            x = np.arange(len(plot_df))
            width = 0.38
            plt.figure(figsize=(11, 5))
            plt.bar(x - width/2, plot_df["before_mean"], width, label="Before fill")
            plt.bar(x + width/2, plot_df["after_mean"], width, label="After fill")
            plt.xticks(x, plot_df["variable"], rotation=30)
            plt.title("Mean trước và sau imputation")
            plt.ylabel("Mean")
            plt.legend()
            save_fig("15_mean_before_after_imputation.png")

    # Std shift
    if {"variable", "before_std", "after_std"}.issubset(dist_shift.columns):
        plot_df = dist_shift.dropna(subset=["before_std", "after_std"]).copy()
        if not plot_df.empty:
            x = np.arange(len(plot_df))
            width = 0.38
            plt.figure(figsize=(11, 5))
            plt.bar(x - width/2, plot_df["before_std"], width, label="Before fill")
            plt.bar(x + width/2, plot_df["after_std"], width, label="After fill")
            plt.xticks(x, plot_df["variable"], rotation=30)
            plt.title("Standard deviation trước và sau imputation")
            plt.ylabel("Std")
            plt.legend()
            save_fig("16_std_before_after_imputation.png")
else:
    display_section_note(
        "Chưa tìm thấy report trước/sau imputation. Hãy chạy Step 5 improved để tạo "
        "`distribution_before_after_imputation.csv`."
    )


In [ ]:

# 7.3 Source coverage: tỷ lệ dữ liệu đến từ GSOD/ERA5/Interpolation
source_path = SILVER_REPORT_DIR / "value_source_coverage.csv"
if source_path.exists():
    source_coverage = pd.read_csv(source_path)
    display(source_coverage)
    save_table(source_coverage, "17_value_source_coverage_copy.csv")

    if {"variable", "source", "rate"}.issubset(source_coverage.columns):
        pivot = source_coverage.pivot_table(index="variable", columns="source", values="rate", fill_value=0)
        display(pivot)
        plt.figure(figsize=(10, 5))
        bottom = np.zeros(len(pivot))
        for col in pivot.columns:
            plt.bar(pivot.index, pivot[col] * 100, bottom=bottom * 100, label=col)
            bottom += pivot[col].values
        plt.ylabel("Rate (%)")
        plt.title("Nguồn dữ liệu sau imputation theo biến")
        plt.xticks(rotation=30)
        plt.legend()
        save_fig("17_value_source_coverage.png")
else:
    display_section_note("Chưa tìm thấy value_source_coverage.csv từ Step 5.")



**Nhận xét cần viết trong báo cáo:**  
ERA5 không cần giống GSOD tuyệt đối vì một nguồn là lưới tái phân tích, một nguồn là quan trắc điểm. Điều cần chứng minh là ERA5 không tạo sai lệch phân phối nghiêm trọng khi dùng để bù khuyết. Vì vậy nên báo cáo correlation, bias, MAE/RMSE và mean/std trước-sau fill.


## 8. Correlation heatmap và quan hệ đa chiều

In [ ]:

# Xác định feature số hợp lệ cho EDA/model
LEAKAGE_COLS = [
    TARGET_COL, "PRCP", "PRCP_mm", "target_prcp_mm", "target_time", "DATE"
]
ID_TIME_LOCATION_COLS = [
    "time", "STATION"
]
# Có thể giữ LAT/LON/ELEVATION cho EDA, nhưng thường loại khỏi model chính để tránh học thuộc trạm.
LOCATION_COLS = ["LATITUDE", "LONGITUDE", "ELEVATION"]

EXCLUDE_COLS = set([c for c in LEAKAGE_COLS + ID_TIME_LOCATION_COLS if c in df.columns])

numeric_features = [
    col for col in df.columns
    if col not in EXCLUDE_COLS
    and not col.endswith("_source")
    and pd.api.types.is_numeric_dtype(df[col])
]

print(f"Số numeric feature hợp lệ cho EDA/model: {len(numeric_features)}")
print(numeric_features[:80])

pd.Series(numeric_features, name="numeric_features").to_csv(TABLE_DIR / "18_numeric_features_for_analysis.csv", index=False)


In [ ]:

# Correlation heatmap trên top biến có variance cao để tránh hình quá rối
if len(numeric_features) > 1:
    # Impute tạm cho correlation EDA, không dùng cho train final.
    corr_df = df[numeric_features].copy()
    variances = corr_df.var(numeric_only=True).sort_values(ascending=False)
    top_corr_features = variances.head(min(35, len(variances))).index.tolist()

    corr = corr_df[top_corr_features].corr(method="spearman")
    corr.to_csv(TABLE_DIR / "19_spearman_correlation_matrix_top_features.csv")

    plt.figure(figsize=(13, 10))
    if HAS_SEABORN:
        sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
    else:
        plt.imshow(corr.values, aspect="auto")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
        plt.colorbar(label="Spearman correlation")
    plt.title("Spearman correlation heatmap — top numeric features")
    save_fig("19_spearman_correlation_heatmap.png")
else:
    print("Không đủ numeric feature để vẽ correlation heatmap.")


In [ ]:

# Spearman correlation với target
target_corr_rows = []
for col in numeric_features:
    pair = df[[col, TARGET_COL]].dropna()
    if pair[col].nunique() <= 1 or pair[TARGET_COL].nunique() <= 1:
        continue
    corr_val = pair[col].corr(pair[TARGET_COL], method="spearman")
    target_corr_rows.append({"feature": col, "spearman_corr_with_target": corr_val, "abs_corr": abs(corr_val)})

target_corr = pd.DataFrame(target_corr_rows).sort_values("abs_corr", ascending=False)
display(target_corr.head(25))
save_table(target_corr, "20_spearman_correlation_with_target.csv")

if not target_corr.empty:
    plot_df = target_corr.head(20).sort_values("abs_corr", ascending=True)
    plt.figure(figsize=(9, 7))
    plt.barh(plot_df["feature"], plot_df["spearman_corr_with_target"])
    plt.title("Top Spearman correlation với target")
    plt.xlabel("Spearman correlation")
    save_fig("20_spearman_corr_with_target_top20.png")



## 9. Khả năng phân tách lớp bằng histogram/boxplot

Phần này giữ lại ý tưởng từ notebook đính kèm: tính điểm chênh lệch trung bình chuẩn hóa giữa lớp mưa và không mưa, sau đó vẽ các feature tách lớp tốt nhất.


In [ ]:

sep_rows = []
for col in numeric_features:
    g0 = df.loc[df[TARGET_COL] == 0, col].dropna()
    g1 = df.loc[df[TARGET_COL] == 1, col].dropna()
    if len(g0) == 0 or len(g1) == 0:
        continue
    pooled_std = df[col].std(skipna=True)
    score = abs(g1.mean() - g0.mean()) / (pooled_std + 1e-9)
    sep_rows.append({
        "feature": col,
        "mean_no_rain": g0.mean(),
        "mean_rain": g1.mean(),
        "separation_score": score
    })

sep_df = pd.DataFrame(sep_rows).sort_values("separation_score", ascending=False)
display(sep_df.head(20))
save_table(sep_df, "21_separation_score.csv")


In [ ]:

TOP_PLOT_N = 12
top_plot_features = sep_df.head(TOP_PLOT_N)["feature"].tolist()

if top_plot_features:
    n_cols = 3
    n_rows = math.ceil(len(top_plot_features) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(top_plot_features):
        ax = axes[i]
        plot_df = df[[TARGET_COL, col]].dropna()
        if HAS_SEABORN:
            sns.kdeplot(data=plot_df, x=col, hue=TARGET_COL, common_norm=False, ax=ax)
        else:
            for cls in sorted(plot_df[TARGET_COL].dropna().unique()):
                ax.hist(plot_df.loc[plot_df[TARGET_COL] == cls, col], bins=40, alpha=0.45, label=str(cls), density=True)
            ax.legend(title=TARGET_COL)
        ax.set_title(col)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Phân phối top feature theo lớp mưa/không mưa", y=1.02)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "22_top_feature_distribution_by_class.png", dpi=300, bbox_inches="tight")
    plt.show()



**Nhận xét cần viết trong báo cáo:**  
Feature có hai phân phối tách biệt giữa lớp 0 và lớp 1 có khả năng giúp mô hình phân loại tốt hơn. Tuy nhiên, trực quan hóa chỉ là bằng chứng ban đầu, cần xác nhận thêm bằng kiểm định thống kê, mutual information và đánh giá mô hình.


## 10. Feature Selection: Mann–Whitney U, Mutual Information và Spearman filtering

In [ ]:

# Chia train/test theo thời gian để tránh leakage trong feature selection
if "time" in df.columns and df["time"].notna().any():
    # Nếu có mốc 2023/2024 thì dùng split giống Step 7.
    train_mask = df["time"] < pd.Timestamp("2023-01-01")
    test_mask = df["time"] >= pd.Timestamp("2024-01-01")
    if train_mask.sum() == 0 or test_mask.sum() == 0:
        unique_dates = np.array(sorted(df["time"].dropna().unique()))
        cutoff_date = unique_dates[int(len(unique_dates) * 0.8)]
        train_mask = df["time"] < cutoff_date
        test_mask = df["time"] >= cutoff_date
        print("Fallback cutoff date:", cutoff_date)
    else:
        print("Time split: train < 2023-01-01; test >= 2024-01-01")
else:
    train_mask = np.arange(len(df)) < int(len(df) * 0.8)
    test_mask = ~train_mask
    print("Không có time, dùng split 80/20 theo thứ tự dòng.")

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

X_train_raw = train_df[numeric_features]
y_train = train_df[TARGET_COL].astype(int)
X_test_raw = test_df[numeric_features]
y_test = test_df[TARGET_COL].astype(int)

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=numeric_features,
    index=X_train_raw.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=numeric_features,
    index=X_test_raw.index
)

print("Train shape:", X_train_imp.shape, "Test shape:", X_test_imp.shape)
print("Train rain rate:", y_train.mean(), "Test rain rate:", y_test.mean())


In [ ]:

# Mann–Whitney U trên train
mw_rows = []
for col in numeric_features:
    x0 = X_train_imp.loc[y_train == 0, col]
    x1 = X_train_imp.loc[y_train == 1, col]
    if x0.nunique() <= 1 and x1.nunique() <= 1:
        continue
    stat, p_value = mannwhitneyu(x0, x1, alternative="two-sided")
    mw_rows.append({
        "feature": col,
        "u_statistic": stat,
        "p_value": p_value,
        "minus_log10_p": -np.log10(max(p_value, 1e-300)),
        "mann_whitney_keep": p_value < 0.05
    })

mw_df = pd.DataFrame(mw_rows).sort_values("p_value")
display(mw_df.head(25))
save_table(mw_df, "23_mann_whitney_results.csv")

# Mutual Information trên train
mi_scores = mutual_info_classif(X_train_imp, y_train, random_state=108)
mi_df = pd.DataFrame({
    "feature": numeric_features,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)

display(mi_df.head(25))
save_table(mi_df, "24_mutual_information_results.csv")


In [ ]:

# Gộp điểm Mann–Whitney và MI
def minmax(s):
    s = pd.Series(s).astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

score_df = (
    mw_df[["feature", "p_value", "minus_log10_p", "mann_whitney_keep"]]
    .merge(mi_df, on="feature", how="outer")
    .fillna(0)
)

score_df["score"] = (
    0.5 * minmax(score_df["minus_log10_p"]) +
    0.5 * minmax(score_df["mutual_information"])
)

score_df = score_df.sort_values("score", ascending=False)
display(score_df.head(25))
save_table(score_df, "25_combined_filter_score.csv")


In [ ]:

# Spearman correlation filtering
CORR_THRESHOLD = 0.90

candidate_order = score_df["feature"].tolist()
corr = X_train_imp[candidate_order].corr(method="spearman").abs()

selected_after_corr = []
dropped_corr = []

for feature in candidate_order:
    if feature not in corr.columns:
        continue

    high_corr_with = None
    for kept in selected_after_corr:
        if corr.loc[feature, kept] > CORR_THRESHOLD:
            high_corr_with = kept
            break

    if high_corr_with is not None:
        dropped_corr.append({
            "dropped_feature": feature,
            "kept_feature": high_corr_with,
            "spearman_abs_corr": corr.loc[feature, high_corr_with]
        })
    else:
        selected_after_corr.append(feature)

corr_drop_df = pd.DataFrame(dropped_corr)
print(f"Số feature sau lọc tương quan: {len(selected_after_corr)} / {len(candidate_order)}")
display(corr_drop_df.head(25))
save_table(corr_drop_df, "26_dropped_by_correlation.csv")
pd.Series(selected_after_corr, name="selected_after_corr").to_csv(TABLE_DIR / "27_selected_after_correlation.csv", index=False)


## 11. Random Forest importance, chọn feature cuối cùng và đánh giá nhanh

In [ ]:

# Random Forest importance trên feature đã lọc correlation
rf_features = selected_after_corr

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=108,
    n_jobs=-1
)

rf.fit(X_train_imp[rf_features], y_train)

rf_imp_df = pd.DataFrame({
    "feature": rf_features,
    "rf_importance": rf.feature_importances_
}).sort_values("rf_importance", ascending=False)

display(rf_imp_df.head(30))
save_table(rf_imp_df, "28_random_forest_importance_from_eda.csv")

plt.figure(figsize=(10, 8))
plot_df = rf_imp_df.head(25).sort_values("rf_importance", ascending=True)
plt.barh(plot_df["feature"], plot_df["rf_importance"])
plt.title("Top 25 feature importance — Random Forest EDA")
plt.xlabel("Importance")
plt.ylabel("")
save_fig("28_random_forest_importance_from_eda.png")


In [ ]:

# Tập feature cuối cùng: kết hợp filter score và Random Forest importance
TOP_K = 20

final_score_df = (
    score_df[["feature", "score", "p_value", "mutual_information"]]
    .merge(rf_imp_df, on="feature", how="inner")
)

final_score_df["final_score"] = (
    0.4 * minmax(final_score_df["score"]) +
    0.6 * minmax(final_score_df["rf_importance"])
)

final_score_df = final_score_df.sort_values("final_score", ascending=False)
final_features = final_score_df.head(min(TOP_K, len(final_score_df)))["feature"].tolist()

print(f"Số feature cuối cùng: {len(final_features)}")
print(final_features)

save_table(final_score_df, "29_final_feature_ranking.csv")
pd.Series(final_features, name="final_selected_features").to_csv(
    TABLE_DIR / "30_final_selected_features.csv", index=False
)


In [ ]:

def evaluate_feature_set(name, features):
    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=108,
            n_jobs=-1
        ))
    ])

    pipe.fit(X_train_raw[features], y_train)
    pred = pipe.predict(X_test_raw[features])
    proba = pipe.predict_proba(X_test_raw[features])[:, 1]

    return {
        "feature_set": name,
        "n_features": len(features),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba) if y_test.nunique() > 1 else np.nan,
        "pr_auc": average_precision_score(y_test, proba) if y_test.nunique() > 1 else np.nan,
        "mcc": matthews_corrcoef(y_test, pred)
    }

results = []
results.append(evaluate_feature_set("All numeric valid features", numeric_features))
results.append(evaluate_feature_set("After Spearman correlation filtering", selected_after_corr))
results.append(evaluate_feature_set(f"Top {len(final_features)} final features", final_features))

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)
save_table(results_df, "31_feature_set_model_comparison.csv")


In [ ]:

# Xuất dữ liệu selected features phục vụ phụ lục/báo cáo
metadata_cols = [c for c in ["STATION", "time", "LATITUDE", "LONGITUDE", "ELEVATION"] if c in df.columns]
keep_cols = metadata_cols + [TARGET_COL] + final_features

cleaned_selected_df = df[keep_cols].copy()
model_ready_selected_df = df[[TARGET_COL] + final_features].copy()

cleaned_selected_path = OUTPUT_DIR / "cleaned_weather_data_selected_features.csv"
model_ready_selected_path = OUTPUT_DIR / "model_ready_selected_features.csv"

cleaned_selected_df.to_csv(cleaned_selected_path, index=False)
model_ready_selected_df.to_csv(model_ready_selected_path, index=False)

print("Đã xuất:")
print("-", cleaned_selected_path)
print("-", model_ready_selected_path)
display(cleaned_selected_df.head())


## 12. Top feature importance sau model validation

In [ ]:

# Đọc feature importance từ Step 7 nếu có
importance_candidates = [
    MODEL_REPORT_DIR / "feature_importance_lightgbm.csv",
    MODEL_REPORT_DIR / "feature_importance_randomforest.csv",
    TABLE_DIR / "28_random_forest_importance_from_eda.csv",
]

imp_path = first_existing(importance_candidates)
if imp_path:
    model_imp = pd.read_csv(imp_path)
    print("Loaded feature importance:", imp_path)
    display(model_imp.head(30))

    # Chuẩn hóa tên cột importance
    if "importance" in model_imp.columns:
        imp_col = "importance"
    elif "rf_importance" in model_imp.columns:
        imp_col = "rf_importance"
    else:
        imp_col = [c for c in model_imp.columns if "importance" in c.lower()][0]

    plot_df = model_imp.sort_values(imp_col, ascending=False).head(25).sort_values(imp_col, ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(plot_df["feature"], plot_df[imp_col])
    plt.title("Top feature importance sau model validation")
    plt.xlabel("Importance")
    save_fig("32_top_feature_importance_after_model.png")
else:
    display_section_note("Chưa tìm thấy feature importance từ Step 7. Notebook dùng Random Forest importance ở phần EDA làm fallback.")


In [ ]:

# Đọc metrics Step 7 nếu có
metrics_path = MODEL_REPORT_DIR / "model_metrics_test_ranked.csv"
if metrics_path.exists():
    model_metrics = pd.read_csv(metrics_path)
    display(model_metrics)
    save_table(model_metrics, "33_model_metrics_test_ranked_copy.csv")

    metric_cols = [c for c in ["model", "precision_rain", "recall_rain", "f1_rain", "roc_auc", "pr_auc", "brier_score"] if c in model_metrics.columns]
    display(model_metrics[metric_cols])
else:
    display_section_note("Chưa tìm thấy model_metrics_test_ranked.csv từ Step 7. Hãy chạy Step 7 improved để bổ sung bảng này.")



## 13. Kết luận Data Storytelling cho báo cáo

Bạn có thể viết kết luận EDA theo khung sau:

1. **Chất lượng dữ liệu:** dữ liệu có cấu trúc theo trạm và thời gian; cần kiểm soát duplicate, missingness theo biến/trạm/tháng.
2. **Missingness:** nếu missing tập trung theo trạm hoặc tháng, dữ liệu thiếu có tính cấu trúc, do đó dùng ERA5 để bù khuyết hợp lý hơn xóa dòng.
3. **Imputation:** so sánh GSOD–ERA5 và trước/sau imputation giúp chứng minh việc bù khuyết không làm méo phân phối nghiêm trọng.
4. **Mùa vụ:** xác suất mưa thay đổi theo tháng và theo trạm, củng cố việc tạo đặc trưng thời gian chu kỳ.
5. **Feature signal:** Mann–Whitney U, Mutual Information, Spearman và Random Forest importance cho thấy một số biến khí tượng có khả năng phân tách lớp mưa/không mưa.
6. **Leakage control:** các biến liên quan trực tiếp đến lượng mưa gốc như `PRCP`, `PRCP_mm`, `target_prcp_mm` không được đưa vào input model; target duy nhất là `rain_target`.
7. **Vai trò model:** model chỉ dùng để kiểm chứng tính hữu ích của dataset sau tiền xử lý, không phải đóng góp chính của đồ án.

> Câu chốt gợi ý:  
> Quy trình EDA cho thấy bộ dữ liệu sau tích hợp đa nguồn có tính mùa vụ rõ ràng, tồn tại khác biệt không gian giữa các trạm và chứa các tín hiệu khí tượng có khả năng phân biệt ngày mưa/không mưa. Các kiểm tra missingness, so sánh GSOD–ERA5 và phân tích trước/sau imputation cho thấy quá trình bù khuyết được kiểm soát thay vì xử lý cơ học. Tập đặc trưng cuối cùng được lựa chọn dựa trên sự kết hợp giữa ý nghĩa thống kê, phụ thuộc phi tuyến với target, kiểm soát đa cộng tuyến và độ quan trọng trong mô hình kiểm chứng.


In [ ]:

# Tổng hợp nhanh các output chính
summary_outputs = {
    "data_path": str(DATA_PATH),
    "eda_output_dir": str(OUTPUT_DIR),
    "plot_dir": str(PLOT_DIR),
    "table_dir": str(TABLE_DIR),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "target_col": TARGET_COL,
    "prcp_col": PRCP_COL,
    "n_numeric_features_for_analysis": len(numeric_features),
    "n_final_selected_features": len(final_features) if "final_features" in globals() else None,
    "final_selected_features": final_features if "final_features" in globals() else None,
}

with open(OUTPUT_DIR / "eda_quality_report_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_outputs, f, ensure_ascii=False, indent=2)

summary_outputs
